In [5]:
import pandas as pd

# Load the Excel file to inspect sheet names and columns
excel_path = '/home/klby/Documents/DataScrapeV2/CollegeScorecardDataDictionary.xlsx'
output_csv = '/home/klby/Documents/DataScrapeV2/all_api_fields.csv'
xlsx = pd.ExcelFile(excel_path)

print("Sheet names:", xlsx.sheet_names)

# Parse the "data_dictionary" sheet (usually this contains the definitions)
# If that name doesn't exist, I'll check the output of sheet_names
for sheet in xlsx.sheet_names:
    if 'data' in sheet.lower() and 'dictionary' in sheet.lower():
        df = pd.read_excel(excel_path, sheet_name=sheet)
        print(f"\n--- Columns in {sheet} ---")
        print(df.columns.tolist())
        print(f"Row count: {len(df)}")
        break

Sheet names: ['README', 'ChangeLog', 'Glossary', 'Institution_Data_Dictionary', 'Institution_Cohort_Map', 'Most_Recent_Inst_Cohort_Map', 'FieldOfStudy_Data_Dictionary', 'FieldOfStudy_Cohort_Map']

--- Columns in Institution_Data_Dictionary ---
['NAME OF DATA ELEMENT', 'dev-category', 'developer-friendly name', 'API data type', 'INDEX', 'VARIABLE NAME', 'VALUE', 'LABEL', 'SOURCE', 'SHOWN/USE ON SITE', 'NOTES']
Row count: 3567


In [7]:
target_sheet = next((s for s in xlsx.sheet_names if 'data' in s.lower() and 'dictionary' in s.lower()), None)

if target_sheet:
    df = pd.read_excel(excel_path, sheet_name=target_sheet)
    
    # Select relevant columns for documentation
    # Just picking the most common standard names; might need adjustment based on exact file version
    cols_to_keep = ['VARIABLE NAME', 'developer-friendly name', 'NAME OF DATA ELEMENT', 'SOURCE', 'API DATA TYPE', 'dev-category']
    available_cols = [c for c in cols_to_keep if c in df.columns]
    
    if 'VARIABLE NAME' in df.columns:
        # Filter out empty variable names
        clean_df = df[df['VARIABLE NAME'].notna()][available_cols]
        
        # Save to CSV
        clean_df.to_csv(output_csv, index=False)
        
        print(f"✅ Successfully extracted {len(clean_df)} fields.")
        print(f"📝 Full list saved to: {output_csv}")
        print("\n--- Field Summary by Category (Top 10) ---")
        if 'dev-category' in clean_df.columns:
            print(clean_df['dev-category'].value_counts().head(10))
        
        print("\n--- Sample Fields ---")
        print(clean_df[available_cols[:3]].head(5).to_string(index=False))
    else:
        print("❌ Could not find 'VARIABLE NAME' column in the dictionary.")
else:
    print("❌ Could not find a 'data_dictionary' sheet.")

✅ Successfully extracted 3305 fields.
📝 Full list saved to: /home/klby/Documents/DataScrapeV2/all_api_fields.csv

--- Field Summary by Category (Top 10) ---
dev-category
completion    1367
repayment     1094
academics      247
earnings       183
student        131
aid            111
cost            85
school          49
admissions      32
root             6
Name: count, dtype: int64

--- Sample Fields ---
VARIABLE NAME developer-friendly name           NAME OF DATA ELEMENT
       UNITID                      id        Unit ID for institution
        OPEID                 ope8_id 8-digit OPE ID for institution
       OPEID6                 ope6_id 6-digit OPE ID for institution
       INSTNM                    name               Institution name
         CITY                    city                           City


In [8]:
fos_sheet = 'FieldOfStudy_Data_Dictionary'
fos_output_csv = '/home/klby/Documents/DataScrapeV2/field_of_study_api_fields.csv'

if fos_sheet in xlsx.sheet_names:
    print(f"\nProcessing {fos_sheet}...")
    df_fos = pd.read_excel(excel_path, sheet_name=fos_sheet)
    
    print(f"Columns: {df_fos.columns.tolist()}")
    
    # Select relevant columns
    # Adjusting based on inspection, usually they match
    cols_to_keep = ['VARIABLE NAME', 'developer-friendly name', 'NAME OF DATA ELEMENT', 'SOURCE', 'API DATA TYPE', 'dev-category']
    available_cols = [c for c in cols_to_keep if c in df_fos.columns]
    
    if 'VARIABLE NAME' in df_fos.columns:
        clean_df_fos = df_fos[df_fos['VARIABLE NAME'].notna()][available_cols]
        clean_df_fos.to_csv(fos_output_csv, index=False)
        
        print(f"✅ Successfully extracted {len(clean_df_fos)} Field of Study fields.")
        print(f"📝 Full list saved to: {fos_output_csv}")
        
        print("\n--- Field Summary by Category (Top 10) ---")
        if 'dev-category' in clean_df_fos.columns:
            print(clean_df_fos['dev-category'].value_counts().head(10))
            
        print("\n--- Sample Fields ---")
        print(clean_df_fos[available_cols[:3]].head(5).to_string(index=False))
    else:
        print("❌ VARIABLE NAME column not found.")
else:
    print(f"❌ {fos_sheet} not found.")


Processing FieldOfStudy_Data_Dictionary...
Columns: ['NAME OF DATA ELEMENT', 'dev-category', 'developer-friendly name', 'API data type', 'VARIABLE NAME', 'VALUE', 'LABEL', 'SOURCE', 'NOTES', 'INDEX']
✅ Successfully extracted 174 Field of Study fields.
📝 Full list saved to: /home/klby/Documents/DataScrapeV2/field_of_study_api_fields.csv

--- Field Summary by Category (Top 10) ---
dev-category
programs    174
Name: count, dtype: int64

--- Sample Fields ---
VARIABLE NAME        developer-friendly name           NAME OF DATA ELEMENT
       UNITID            cip_4_digit.unit_id        Unit ID for institution
       OPEID6            cip_4_digit.ope6_id 6-digit OPE ID for institution
       INSTNM        cip_4_digit.school.name               Institution name
      CONTROL        cip_4_digit.school.type         Control of institution
         MAIN cip_4_digit.school.main_campus           Flag for main campus
